# FT-Transformer: score one employee (interactive demo)

This companion notebook lets you type in the 17 approved predictors for a single employee and
see the FT-Transformer score. It is separate from the saved-review notebook
(`05_m1_ft_transformer.ipynb`), which is tested never to load or train a model.

**Read this before trusting any number it prints.**

- The formal nested evaluation found **near-chance discrimination** (pooled ROC-AUC 0.498, AP 0.0986
  against a prevalence of 0.1001). The score below carries essentially no information about whether a
  real employee will resign.
- The model used here is a **demo model**: one fixed candidate (FT01) trained by
  `scripts/train_ft_demo_predictor.py` on the 80,000 **development** rows only. It is **not** the
  frozen deployment model, and the formal holdout was never read.
- The score is a raw model output, not a calibrated probability, and 0.50 is not a retention policy.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/feature_contract.yaml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.evaluation.inputs import load_development_data
from src.models.ft_transformer import FTTransformerAdapter

pd.set_option("display.max_columns", None)
DEMO_DIR = ROOT / "results/models/ft_transformer/demo_predictor"

contract = yaml.safe_load((ROOT / "configs/feature_contract.yaml").read_text(encoding="utf-8"))
features = contract["primary_model_features"]
numerical = [item["name"] for item in contract["numerical_features"]]
categorical = {item["name"]: item["expected_categories"] for item in contract["categorical_features"]}

# Development rows only (holdout rows are excluded before parsing). Used for defaults and valid ranges.
development, _ = load_development_data(ROOT)
print("Development rows used for defaults/ranges:", development.shape)
prevalence = float(development.Resigned.mean())
ranges = development[numerical].agg(["min", "max"]).T
defaults = {**development[numerical].median().to_dict(),
            **{name: development[name].mode().iloc[0] for name in categorical}}

if (DEMO_DIR / "ft_transformer_weights.pt").is_file():
    adapter = FTTransformerAdapter.load(DEMO_DIR)
    print("Loaded demo model from", DEMO_DIR.relative_to(ROOT))
else:
    adapter = None
    print("Demo model not found. Create it once with:  python scripts/train_ft_demo_predictor.py")
display(ranges)

## Enter an employee and score

Edit the values in `INPUTS` and re-run the next cell. Categorical values must match the approved
categories exactly. Numerical values outside the range seen in the development data are still
scored but flagged, because the model has never seen such inputs and the result is an extrapolation.

In [ ]:
INPUTS = dict(defaults)          # start from development medians / most common categories
INPUTS.update(
    Age=35,
    Years_At_Company=3,
    Performance_Score=3,
    Monthly_Salary=6000,
    Work_Hours_Per_Week=45,
    Projects_Handled=20,
    Overtime_Hours=10,
    Sick_Days=5,
    Remote_Work_Frequency=50,
    Team_Size=10,
    Training_Hours=50,
    Promotions=1,
    Employee_Satisfaction_Score=3.0,
    Department="IT",
    Gender="Female",
    Job_Title="Analyst",
    Education_Level="Bachelor",
)

def predict_resignation(**inputs):
    if adapter is None:
        raise RuntimeError("Demo model missing. Run: python scripts/train_ft_demo_predictor.py")
    unknown = set(inputs) - set(features)
    missing = set(features) - set(inputs)
    if unknown or missing:
        raise ValueError(f"Unknown inputs: {sorted(unknown)}; missing inputs: {sorted(missing)}")
    for name, allowed in categorical.items():
        if inputs[name] not in allowed:
            raise ValueError(f"{name}={inputs[name]!r} is not one of {allowed}")
    warnings = []
    for name in numerical:
        value = float(inputs[name])
        low, high = ranges.loc[name, "min"], ranges.loc[name, "max"]
        if not low <= value <= high:
            warnings.append(f"{name}={value:g} is outside the development range [{low:g}, {high:g}]")
    row = pd.DataFrame([{name: inputs[name] for name in features}])
    row[numerical] = row[numerical].astype(float)
    return float(adapter.predict_proba(row)[0]), warnings

if adapter is not None:
    score, warnings = predict_resignation(**INPUTS)
    display(pd.DataFrame([INPUTS]).T.rename(columns={0: "Value"}))
    for message in warnings:
        print("WARNING:", message)
    display(Markdown(
        f"**Model score: {score:.4f}** (development prevalence {prevalence:.4f}; "
        f"descriptive 0.50 threshold -> {'would be flagged' if score >= 0.5 else 'would not be flagged'}).\n\n"
        "*This model's discrimination is near chance, so this score is a demonstration of the "
        "interface, not a usable prediction of resignation.*"))

## Sanity check on real development rows

Three development employees scored through the same helper, with their recorded outcome. Because
discrimination is near chance, scores for those who resigned will look just like the others.

In [ ]:
if adapter is not None:
    sample = development.sample(3, random_state=0)
    rows = []
    for _, employee in sample.iterrows():
        score, _ = predict_resignation(**{name: employee[name] for name in features})
        rows.append({"Employee_ID": int(employee.Employee_ID), "Recorded Resigned": int(employee.Resigned),
                     "Model score": round(score, 4)})
    display(pd.DataFrame(rows))